# StudyBuddy Agent — Agentic AI Application

## Overview

StudyBuddy is an agentic AI assistant designed to help students study efficiently.
It demonstrates core agentic AI concepts including prompting, memory management,
tool usage, agent loops, and structured action parsing.

The agent can:

- Generate quizzes on a given topic
- Summarize provided text
- Manage a simple study task list
- Maintain conversational memory across interactions
- Select and execute tools based on user intent

The system is implemented using LiteLLM with Groq as the LLM provider and follows
a tool-driven agent architecture.

---

## Features

- Agent loop with memory
- Structured tool invocation using JSON actions
- Custom tools:
  - `generate_quiz`
  - `summarize_text`
  - `add_task`
  - `get_tasks`
- Robust parsing and error handling
- Multiple interaction support

---

## Setup Instructions

### 1. Open the notebook in Google Colab

Create a new Colab notebook and copy all provided code cells.

---

### 2. Install dependencies

Run:

```python
!pip install litellm
```
### 3. Configure Groq API Key

Create a Groq API key from:

https://console.groq.com

Then add in Colab:
```python
import os
os.environ["GROQ_API_KEY"] = "gsk_your_key_here"
```

### 4. Run all cells

Execute each cell from top to bottom.

How to Use

Start the agent:
```python
agent_loop()
```
Generate Quiz
```
Make me a 6-question quiz on Information Security basics, medium difficulty
```
Summarize Text
```
Summarize this: <paste your notes>
```
Add Task
```
Add task: Review CIA triad and access control models
```

Show Tasks
```
Show my tasks
```
Exit Agent
```
exit
```


In [ ]:
!pip install -q litellm

import os
import json
from typing import List, Dict, Any
from litellm import completion

os.environ.pop("OPENAI_API_KEY", None)
os.environ["GROQ_API_KEY"] = "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"

DEFAULT_MODEL = "groq/llama-3.1-8b-instant"

# 1️⃣ GENERATE RESPONSE FUNCTION

def generate_response(messages, tools=None):
    return completion(
        model=DEFAULT_MODEL,
        messages=messages,
        tools=tools,
        max_tokens=1024
    )

# 2️⃣ DEFINE GOALS

class Goal:
    def __init__(self, priority, name, description):
        self.priority = priority
        self.name = name
        self.description = description

# 3️⃣ TOOL FUNCTIONS

TASKS = []

def summarize_text(text: str) -> str:
    return text[:300] + "..." if len(text) > 300 else text

def generate_quiz(topic: str) -> List[str]:
    return [
        f"What is {topic}?",
        f"Explain key concepts of {topic}.",
        f"Why is {topic} important?"
    ]

def add_task(task: str) -> bool:
    TASKS.append(task)
    return True

def get_tasks() -> List[str]:
    return TASKS.copy()

def terminate(message: str) -> str:
    return message


# 4️⃣ ACTION REGISTRY

class Action:
    def __init__(self, name, function, description, parameters, terminal=False):
        self.name = name
        self.function = function
        self.description = description
        self.parameters = parameters
        self.terminal = terminal


class ActionRegistry:
    def __init__(self):
        self.actions = {}

    def register(self, action: Action):
        self.actions[action.name] = action

    def get_tools_schema(self):
        tools = []
        for action in self.actions.values():
            tools.append({
                "type": "function",
                "function": {
                    "name": action.name,
                    "description": action.description,
                    "parameters": action.parameters
                }
            })
        return tools


# 5️⃣ AGENT CLASS

class Agent:

    def __init__(self, goals, action_registry):
        self.goals = goals
        self.registry = action_registry
        self.memory = []

    def run(self, user_input, max_iterations=10):

        self.memory.append({"role": "user", "content": user_input})

        for _ in range(max_iterations):

            response = generate_response(
                messages=self.memory,
                tools=self.registry.get_tools_schema()
            )

            message = response.choices[0].message

            # If tool call happens
            if message.tool_calls:

                tool_call = message.tool_calls[0]
                tool_name = tool_call.function.name
                args = json.loads(tool_call.function.arguments) if tool_call.function.arguments else {}

                print(f"\nTool Called: {tool_name}")
                print("Arguments:", args)

                action = self.registry.actions[tool_name]
                result = action.function(**args) if args else action.function()

                print("Tool Result:", result)

                # Add tool result
                self.memory.append({
                    "role": "tool",
                    "tool_call_id": tool_call.id,
                    "content": json.dumps(result)
                })

                # If terminal action → stop
                if action.terminal:
                    print("\nAgent:", result)
                    break

            else:
                print("\nAgent:", message.content)
                self.memory.append({
                    "role": "assistant",
                    "content": message.content
                })

                break

# 6️⃣ MAIN

def main():

    goals = [
        Goal(1, "Assist Study Tasks", "Help with quizzes, summaries, and task tracking"),
        Goal(2, "Terminate", "End session when requested")
    ]

    registry = ActionRegistry()

    registry.register(Action(
        name="summarize_text",
        function=summarize_text,
        description="Summarizes a given text.",
        parameters={
            "type": "object",
            "properties": {
                "text": {"type": "string"}
            },
            "required": ["text"]
        }
    ))

    registry.register(Action(
        name="generate_quiz",
        function=generate_quiz,
        description="Generates quiz questions on a topic.",
        parameters={
            "type": "object",
            "properties": {
                "topic": {"type": "string"}
            },
            "required": ["topic"]
        }
    ))

    registry.register(Action(
        name="add_task",
        function=add_task,
        description="Adds a study task.",
        parameters={
            "type": "object",
            "properties": {
                "task": {"type": "string"}
            },
            "required": ["task"]
        }
    ))

    registry.register(Action(
        name="get_tasks",
        function=get_tasks,
        description="Returns all saved tasks.",
        parameters={
            "type": "object",
            "properties": {}
        }
    ))

    registry.register(Action(
        name="terminate",
        function=terminate,
        description="Terminates the session.",
        parameters={
            "type": "object",
            "properties": {
                "message": {"type": "string"}
            },
            "required": ["message"]
        },
        terminal=True
    ))

    agent = Agent(goals, registry)

    print("StudyBuddy Agent started.\n")

    while True:
        user_input = input("User: ")

        if user_input.lower() in ["exit", "quit"]:
            print("Session ended.")
            break

        agent.run(user_input)


if __name__ == "__main__":
    main()

StudyBuddy Agent started.

User: hi

Agent: There are several functions you can use to get started:  

You can use 'summarize_text' to Summarizes a given text,  
'generate_quiz' to Generates quiz questions on a topic,  
'add_task' to Adds a study task,  
)get_tasks to Returns all saved tasks,  
or terminate' to Terminates the session.
User: Generate quiz on Machine Learning

Tool Called: generate_quiz
Arguments: {'topic': 'Machine Learning'}
Tool Result: ['What is Machine Learning?', 'Explain key concepts of Machine Learning.', 'Why is Machine Learning important?']

Tool Called: generate_quiz
Arguments: {'topic': 'Machine Learning'}
Tool Result: ['What is Machine Learning?', 'Explain key concepts of Machine Learning.', 'Why is Machine Learning important?']

Tool Called: generate_quiz
Arguments: {'topic': 'Machine Learning'}
Tool Result: ['What is Machine Learning?', 'Explain key concepts of Machine Learning.', 'Why is Machine Learning important?']

Tool Called: generate_quiz
Arguments:

BadRequestError: litellm.BadRequestError: GroqException - {"error":{"message":"Failed to call a function. Please adjust your prompt. See 'failed_generation' for more details.","type":"invalid_request_error","code":"tool_use_failed","failed_generation":"\u003cfunction=generate_quiz\u003e{\"topic\": \"Machine Learning\"}"}}
